# Live demo — Swin UNETR spleen (inference only)

> **Reference code — not runnable as-is.** This notebook needs the trained **exp3 checkpoint** produced on the **Shenkar lab GPU**. Checkpoints are gitignored and are **not distributed with this repository**, and the paths below point at the lab filesystem (`/home/ori.grossman/...`). It also needs the MSD Task09 data and CUDA. It is included as a record of how the demo was run, not as something a reader can execute after cloning.

A short notebook for a **live** demo. It trains nothing — it loads the exp3 checkpoint and runs inference only, so on the lab GPU each cell takes seconds to a minute.

This is a **companion** to `Spleen_Segmentation_SwinUNETR.ipynb` (the full notebook is the complete documentation of the project). Here there are three things that can be run on request:

1. **Segmentation quality** — live Dice on a validation volume.
2. **The explanation** — the attention hook and the attention-on-spleen ratio, with a figure.
3. **The null control** — trained vs untrained, showing that the concentration on the spleen is *learned*.

The notebook is submitted without outputs — the outputs are produced when it is run live. Run it top to bottom (Run All), or cell by cell.

In [ ]:
import os, sys, site, glob
us = site.getusersitepackages()
if us not in sys.path: sys.path.insert(0, us)          # after a kernel restart, re-add user-site
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, CropForegroundd, AsDiscrete, SpatialCrop, ResizeWithPadOrCrop)
from monai.networks.nets import SwinUNETR
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.data import decollate_batch

device    = torch.device("cuda")
PATCH     = (96, 96, 96)
DATA_DIR  = "/home/ori.grossman/nn_final/data/Task09_Spleen"
EXP3_CKPT = "/home/ori.grossman/nn_final/experiments/exp3_cosine_noaug/best.pth"
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# validation preprocessing — identical to the main notebook (RAS, 1.5x1.5x2 mm, HU window, crop)
val_tf = Compose([
    LoadImaged(keys=["image", "label"]), EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 2.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=-57, a_max=164, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image")])

imgs = sorted(f for f in glob.glob(DATA_DIR + "/imagesTr/*.nii.gz") if not os.path.basename(f).startswith("._"))
lbls = sorted(f for f in glob.glob(DATA_DIR + "/labelsTr/*.nii.gz") if not os.path.basename(f).startswith("._"))
val_files = [{"image": i, "label": l} for i, l in zip(imgs, lbls)][-9:]     # the same 9 validation volumes

model = SwinUNETR(img_size=PATCH, in_channels=1, out_channels=2, feature_size=48, use_checkpoint=True).to(device)
model.load_state_dict(torch.load(EXP3_CKPT, map_location=device)); model.eval()
print("loaded exp3 checkpoint | validation volumes:", len(val_files))

## 1) Segmentation quality — live Dice on a validation volume

Pick one validation volume, run `sliding_window_inference`, and measure Dice against the ground truth. This demonstrates the headline number live (exp3 averages **0.9465** across the 9 volumes; per-volume it ranges 0.915–0.966).

In [ ]:
CASE = 0                                    # 0..8 — שנה כדי להראות כל נפח validation
post_pred, post_label = AsDiscrete(argmax=True, to_onehot=2), AsDiscrete(to_onehot=2)
dice_metric = DiceMetric(include_background=False, reduction="mean")

sample = val_tf(val_files[CASE])
x = sample["image"].unsqueeze(0).to(device)
y = sample["label"].unsqueeze(0).to(device)
with torch.no_grad(), torch.autocast("cuda"):
    out = sliding_window_inference(x, PATCH, 4, model, overlap=0.5)
dice_metric(y_pred=[post_pred(o) for o in decollate_batch(out)],
            y=[post_label(o) for o in decollate_batch(y)])
print(f"case {CASE}: Dice = {dice_metric.aggregate().item():.4f}")

## 2) The explanation — attention hook and the ratio

Register a `forward_hook` on the softmax of the deepest `WindowAttention` layer (of 8), crop a 96³ patch around the spleen, and compute the **attention-on-spleen ratio** = mean attention inside the spleen mask divided by mean attention outside it. Above 1 means the model concentrates on the organ. The figure shows CT / ground truth / prediction / attention.

In [ ]:
def attention_heat(m, patch5d):
    """Hook the deepest WindowAttention softmax, return a 96^3 heat-map + the raw tensor shape."""
    wa = [mod for _, mod in m.named_modules() if type(mod).__name__ == "WindowAttention"]
    store = {}
    h = wa[-1].softmax.register_forward_hook(lambda a, b, o: store.__setitem__("a", o.detach()))
    with torch.no_grad(), torch.autocast("cuda"):
        m(patch5d)
    h.remove()
    a = store["a"][0]; s = round(a.shape[-1] ** (1 / 3))          # 216 tokens -> 6^3
    g = a.float().mean(0).mean(0).reshape(s, s, s)                # mean over heads, then queries
    g = (g - g.min()) / (g.max() - g.min() + 1e-8)
    heat = F.interpolate(g[None, None], size=PATCH, mode="trilinear")[0, 0].cpu().numpy()
    return heat, tuple(store["a"].shape)

# crop a 96^3 patch centred on the spleen (same recipe as the main notebook's attention section)
la = sample["label"][0].cpu().numpy()
center = np.argwhere(la > 0).mean(0).round().astype(int).tolist()
ci = ResizeWithPadOrCrop(PATCH)(SpatialCrop(roi_center=center, roi_size=PATCH)(sample["image"]))
cl = ResizeWithPadOrCrop(PATCH)(SpatialCrop(roi_center=center, roi_size=PATCH)(sample["label"]))
ci5 = ci.unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(ci5).argmax(1)[0].cpu().numpy()
heat, ashape = attention_heat(model, ci5)
ct, gt = ci[0].cpu().numpy(), cl[0].cpu().numpy()
ratio = heat[gt > 0].mean() / (heat[gt == 0].mean() + 1e-8)
print("attention tensor:", ashape, "| attention-on-spleen ratio:", round(float(ratio), 2))

z = int(np.argmax(gt.sum((0, 1))))
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(ct[:, :, z], cmap="gray"); ax[0].set_title("CT")
ax[1].imshow(ct[:, :, z], cmap="gray"); ax[1].imshow(np.ma.masked_where(gt[:, :, z] == 0, gt[:, :, z]), cmap="Greens", alpha=.6); ax[1].set_title("ground truth")
ax[2].imshow(ct[:, :, z], cmap="gray"); ax[2].imshow(np.ma.masked_where(pred[:, :, z] == 0, pred[:, :, z]), cmap="Reds", alpha=.6); ax[2].set_title("prediction")
ax[3].imshow(ct[:, :, z], cmap="gray"); ax[3].imshow(heat[:, :, z], cmap="jet", alpha=.5); ax[3].set_title(f"attention (ratio {float(ratio):.2f})")
for a_ in ax: a_.axis("off")
plt.tight_layout(); plt.show()

## 3) Null control — is the concentration learned?

Exactly the same measurement, but on an **untrained** model (random initialisation, same architecture, same patch). If the trained model's ratio is meaningfully higher than the untrained one's, the concentration on the spleen is **learned** — it is not an artifact of the architecture or of the centred crop. (In the full run over all 9 cases: trained ≈ **1.90** vs untrained ≈ **1.11** and a random box ≈ 1.28.)

In [ ]:
untrained = SwinUNETR(img_size=PATCH, in_channels=1, out_channels=2, feature_size=48).to(device).eval()
heat_u, _ = attention_heat(untrained, ci5)               # random init, same architecture, same patch
ratio_u = heat_u[gt > 0].mean() / (heat_u[gt == 0].mean() + 1e-8)
print(f"trained ratio   : {float(ratio):.2f}")
print(f"untrained ratio : {float(ratio_u):.2f}")
print("=> concentration is LEARNED" if ratio > ratio_u + 0.3 else "=> inconclusive on this single patch (see the full 9-case control in the main notebook)")